## Experiment 3: Event detection ResNet-18 - CNN training

In [ ]:
import os
import time
import copy
import torch
import numpy as np
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
from scipy.ndimage import uniform_filter1d
from torch.optim import lr_scheduler
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score

In [ ]:
# Training parameters

DATA_DIR        = "/home/sermomon/PROJECTS/msr-das-cnn-resnet-train001/run001/train"
OUTPUT_DIR      = "/home/sermomon/PROJECTS/msr-das-cnn-resnet-train001/run001/train_results"

MODEL_NAME      = "best.pth"

IMG_SIZE        = (1024, 695)
BATCH_SIZE      = 4
NUM_EPOCHS      = 25
LEARNING_RATE   = 1e-4
WEIGHT_DECAY    = 1e-4
VAL_SPLIT       = 0.25

RESNET_VARIANT  = 18; FEATURE_EXTRACT = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Data transformations

train_transforms = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    # ImageNet normalization parameters
    # transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transforms = transforms.Compose([
    transforms.Resize((681, 1028)),
    transforms.ToTensor(),
    # ImageNet normalization parameters
    # transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [ ]:
# Data loaders

def load_datasets(data_dir, val_split):
    full_dataset = datasets.ImageFolder(root=data_dir)

    n_total = len(full_dataset)
    n_val   = int(n_total * val_split)
    n_train = n_total - n_val

    train_data, val_data = random_split(
        full_dataset, [n_train, n_val], generator=torch.Generator().manual_seed(42))

    train_data.dataset = copy.deepcopy(full_dataset)
    train_data.dataset.transform = train_transforms

    val_dataset = copy.deepcopy(full_dataset)
    val_dataset.transform = val_transforms
    val_data = torch.utils.data.Subset(val_dataset, val_data.indices)

    return train_data, val_data, full_dataset.classes

def build_dataloaders(data_dir, val_split, batch_size):
    train_ds, val_ds, class_names = load_datasets(data_dir, val_split)

    train_loader = DataLoader(
        train_ds, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
    
    val_loader = DataLoader(
        val_ds, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

    print(f"\nClases detectadas: {class_names}")
    print(f"Muestras totales : {len(train_ds) + len(val_ds)}")
    print(f"Training: {len(train_ds)}")
    print(f"Validation: {len(val_ds)}\n")

    return {"train": train_loader, "val": val_loader}, class_names

In [ ]:
# Model: ResNet-18

_RESNET_VARIANTS = {
    18:  models.resnet18,
    34:  models.resnet34,
    50:  models.resnet50,
    101: models.resnet101,
    152: models.resnet152,
}

def build_model(num_classes, variant=50, feature_extract=False):
    if variant not in _RESNET_VARIANTS:
        raise ValueError(f"ResNet {variant} not supported. Available ResNet models: {list(_RESNET_VARIANTS)}")

    model = _RESNET_VARIANTS[variant](weights="IMAGENET1K_V1")

    if feature_extract:
        for param in model.parameters():
            param.requires_grad = False

    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)

    print(f"Model: ResNet-{variant}")
    print(f"Classes: {num_classes}")
    print(f"Mode: {'Feature extraction' if feature_extract else 'Fine-tuning'}")
    print(f"Trainable parameters:"
          f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}\n")

    return model

In [ ]:
# Model training functions

def train_model(model, dataloaders, criterion, optimizer, scheduler, device, num_epochs, output_dir, model_name):
    os.makedirs(output_dir, exist_ok=True)
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0

    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

    for epoch in range(num_epochs):
        print(f"Epoch {epoch+1}/{num_epochs}  {'─'*40}")
        t0 = time.time()

        for phase in ["train", "val"]:
            model.train() if phase == "train" else model.eval()

            running_loss, running_corrects = 0.0, 0

            for inputs, labels in dataloaders[phase]:
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == "train"):
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
                    preds = outputs.argmax(dim=1)

                    if phase == "train":
                        loss.backward()
                        optimizer.step()

                running_loss     += loss.item() * inputs.size(0)
                running_corrects += (preds == labels).sum().item()

            n = len(dataloaders[phase].dataset)
            epoch_loss = running_loss / n
            epoch_acc  = running_corrects / n

            history[f"{phase}_loss"].append(epoch_loss)
            history[f"{phase}_acc"].append(epoch_acc)

            print(f"  {phase:5s}  loss: {epoch_loss:.4f}  acc: {epoch_acc:.4f}")

            if phase == "val" and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())
                torch.save(best_model_wts,
                           os.path.join(output_dir, model_name))
                print(f"Best checkpoint saved (acc={best_acc:.4f})")

        if scheduler is not None:
            scheduler.step()

        print(f"Time: {time.time()-t0:.1f}s\n")

    print(f"Training completed. Best val accuracy: {best_acc:.4f}")
    model.load_state_dict(best_model_wts)
    return model, history

def plot_history(history, output_dir, smooth_factor=0.6):
    
    epochs = np.array(range(1, len(history["train_loss"]) + 1))
    
    def smooth_curve(y, factor=smooth_factor):
        if len(y) < 5:
            return y
        window = max(5, int(len(y) * factor))
        if window % 2 == 0:
            window += 1
        return savgol_filter(y, window_length=window, polyorder=3)
    
    fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
    
    train_color = '#1f77b4'  # Blue
    val_color = '#ff7f0e'    # Orange
    
    plots = [
        {
            'train': history['train_loss'],
            'val': history['val_loss'],
            'ylabel': 'Loss',
            'title': 'Model Loss'
        },
        {
            'train': history['train_acc'],
            'val': history['val_acc'],
            'ylabel': 'Accuracy',
            'title': 'Model Accuracy'
        }
    ]
    
    for ax, config in zip(axes, plots):
        train_data = np.array(config['train'])
        val_data = np.array(config['val'])
        
        ax.plot(epochs, train_data, color=train_color, alpha=0.3, linewidth=1, label='_nolegend_')
        ax.plot(epochs, val_data, color=val_color, alpha=0.3, linewidth=1, label='_nolegend_')
        
        ax.plot(epochs, smooth_curve(train_data), color=train_color, linewidth=3, label='Training')
        ax.plot(epochs, smooth_curve(val_data), color=val_color, linewidth=3, label='Validation')
        
        ax.set_xlabel('Epoch', fontsize=18, fontweight='600')
        ax.set_ylabel(config['ylabel'], fontsize=18, fontweight='600')
        ax.set_title(config['title'], fontsize=20, fontweight='bold', pad=15)
        
        ax.tick_params(axis='both', which='major', labelsize=15, width=1.5, length=6)
        
        ax.legend(fontsize=15, loc='best', frameon=True, shadow=True, fancybox=True, framealpha=0.95)
        
        ax.grid(True, alpha=0.3, linestyle='--', linewidth=0.8)
        ax.set_axisbelow(True)
        
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_linewidth(1.5)
        ax.spines['bottom'].set_linewidth(1.5)
    
    plt.tight_layout()
    
    for fmt, dpi in [('png', 600), ('pdf', None)]:
        path = os.path.join(output_dir, f"training_curves.{fmt}")
        if dpi:
            plt.savefig(path, dpi=dpi, bbox_inches='tight', facecolor='white', pad_inches=0.1)
        else:
            plt.savefig(path, bbox_inches='tight', pad_inches=0.1)
        print(f"Saved: {path}")
    
    plt.show()

In [ ]:
# Run model training

# --- 0. CUDA - GPU ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nDevice: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}\n")

# --- 1. Load data ---
dataloaders, class_names = build_dataloaders(DATA_DIR, VAL_SPLIT, BATCH_SIZE)
num_classes = len(class_names)

# --- 2. Instance model ---
model = build_model(num_classes, variant=RESNET_VARIANT, feature_extract=FEATURE_EXTRACT).to(device)

# --- 3. Loss function ---
criterion = nn.CrossEntropyLoss()
params_to_update = [p for p in model.parameters() if p.requires_grad]
optimizer = optim.AdamW(params_to_update, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

# --- 4. Scheduler ---
scheduler = lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.5)

# --- 5. Train ---
model, history = train_model(
    model, dataloaders, criterion, optimizer, scheduler,
    device, NUM_EPOCHS, OUTPUT_DIR, MODEL_NAME)

# --- 6. Plot training curves ---
plot_history(history, OUTPUT_DIR)

# --- 7. Class mapping ---
mapping_path = os.path.join(OUTPUT_DIR, "Class_mapping.txt")
with open(mapping_path, "w") as f:
    for idx, name in enumerate(class_names):
        f.write(f"{idx}: {name}\n")
print(f"Class mapping saved: {mapping_path}")